# 06 — Full Supervised Baseline — DistilBERT

Trains `utils.config.CLASSIFIER_MODEL_NAME` on 100% of the train split
(full data, `config.CLASSIFIER_SAMPLE_SIZE=None`) — the upper-bound
reference every other method is compared against, 16-way.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_semisupervised
from utils.modeling import get_predictions, train_model
from utils.samples import save_full_output, save_label_samples

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

train_sample = stratified_sample(train_clean, config.CLASSIFIER_SAMPLE_SIZE, seed=config.SEED)
print(f"Training on {len(train_sample)} fully-labeled rows (upper bound baseline)")

Training on 319 fully-labeled rows (upper bound baseline)


In [3]:
model, tokenizer = train_model(train_sample, model_name=config.CLASSIFIER_MODEL_NAME, epochs=3)

test_probs = get_predictions(model, tokenizer, test_clean["text"].tolist())
test_preds = test_probs.argmax(axis=1)

results, report, cm = evaluate_semisupervised(
    test_clean["label"].to_numpy(), test_preds, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_full_supervised.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_full_supervised.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved full-supervised baseline results.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,2.504869


                precision    recall  f1-score   support

ARTS & CULTURE       1.00      0.80      0.89         5
      BUSINESS       0.50      0.60      0.55         5
        COMEDY       0.36      0.80      0.50         5
         CRIME       0.50      1.00      0.67         5
     EDUCATION       0.60      0.60      0.60         5
 ENTERTAINMENT       1.00      0.20      0.33         5
   ENVIRONMENT       0.50      0.20      0.29         5
        HEALTH       1.00      0.40      0.57         5
         MEDIA       0.50      0.80      0.62         5
          NEWS       0.50      0.40      0.44         5
      POLITICS       0.75      0.60      0.67         5
      RELIGION       0.75      0.60      0.67         5
       SCIENCE       0.71      1.00      0.83         5
        SPORTS       0.80      0.80      0.80         5
          TECH       0.67      0.80      0.73         5
         WOMEN       1.00      0.20      0.33         5

      accuracy                           0.61 

In [4]:
save_label_samples(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(),
    config.CLASS_NAMES, confidence=test_probs.max(axis=1), n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_full_supervised.csv")
save_full_output(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(), config.CLASS_NAMES,
    confidence=test_probs.max(axis=1), extra_columns={"summary": test_clean["summary"].tolist()},
    path=config.RESULTS_DIR / "full_labels_full_supervised.csv")
print("Saved sample + full-row outputs for full_supervised.")

Saved sample + full-row outputs for full_supervised.
